# 🎁 Поиск активных официальных розыгрышей Telegram — в облаке

Ничего ставить на компьютер не нужно: всё выполняется на серверах Google Colab, работает и с телефона.
Скрипт **только читает** публичные каналы. Он ни на что не подписывается и ничего не публикует.

**Перед первым запуском (один раз):**
1. Получите `api_id` и `api_hash` на https://my.telegram.org → *API development tools*.
2. *(Удобнее всего)* Нажмите значок 🔑 **«Секреты»** на панели слева, добавьте `TG_API_ID` и `TG_API_HASH` и включите им доступ к блокноту. Если секреты не заданы, блокнот спросит ключи при запуске.

**Запуск:** выполняйте ячейки по очереди кнопкой ▶. При первом входе введите номер телефона, затем код, который Telegram пришлёт в приложение, и пароль 2FA, если он у вас включён.

> Код входа вводите только в поле под ячейкой этого блокнота. Никому его не пересылайте.

In [ ]:
# 1. Установка (≈20 секунд)
import subprocess, sys, urllib.request

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "telethon>=1.45,<2"], check=True)
for ref in ("main", "claude/telegram-giveaway-finder-mhm9ut"):
    url = f"https://raw.githubusercontent.com/JustHappyFox/Test/{ref}/tg_giveaway_finder.py"
    try:
        src = urllib.request.urlopen(url, timeout=30).read()
    except Exception:
        continue
    if b"GiveawayFinder" in src:
        open("tg_giveaway_finder.py", "wb").write(src)
        print("✅ Скрипт загружен (ветка", ref + ")")
        break
else:
    raise RuntimeError("Не удалось скачать tg_giveaway_finder.py с GitHub")

In [ ]:
# 2. Ключи Telegram (из «Секретов» Colab или ввод вручную)
import os, getpass

def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

os.environ["TG_API_ID"] = (os.environ.get("TG_API_ID") or colab_secret("TG_API_ID") or input("api_id: ")).strip()
os.environ["TG_API_HASH"] = (os.environ.get("TG_API_HASH") or colab_secret("TG_API_HASH")
                             or getpass.getpass("api_hash: ")).strip()
if colab_secret("TG_STRING_SESSION"):
    os.environ["TG_STRING_SESSION"] = colab_secret("TG_STRING_SESSION")
    print("✅ Ключи заданы, используется сохранённая сессия — код вводить не нужно")
else:
    print("✅ Ключи заданы")

## Поиск

Настройки задаются в строке `ARGS`. Полезные варианты:

| Что нужно | `ARGS` |
|---|---|
| Искать по всем найденным публичным каналам (без подписок) | `scan --check --careful` |
| То же плюс ваши подписки | `scan --check --careful --dialogs` |
| Только Stars и только там, где вам можно участвовать | `scan --check --careful --prize stars --hide-ineligible` |
| Больше каналов | `scan --check --careful --max-channels 5000` |

`--careful` делает паузу 30 с после каждых 500 каналов и спокойно пережидает ограничения Telegram на частоту запросов (FloodWait).
Обход 500 каналов занимает примерно 10–20 минут. Если ограничение Telegram длиннее `--max-flood-wait`, обход остановится и покажет то, что уже нашёл.

In [ ]:
# 3. Поиск розыгрышей
ARGS = "scan --check --careful"

import importlib, logging, shlex
import tg_giveaway_finder as tgf
importlib.reload(tgf)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)
logging.getLogger("telethon").setLevel(logging.WARNING)

args = tgf.apply_preset(tgf.build_parser().parse_args(
    shlex.split(ARGS) + ["--json", "giveaways.json", "--csv", "giveaways.csv"]))
await tgf.amain(args)

In [ ]:
# 4. Таблица с кликабельными ссылками + скачать CSV (открывается в Excel)
import html, json
from IPython.display import HTML, display

rows = json.load(open("giveaways.json", encoding="utf-8"))
esc = html.escape
trs = []
for r in rows:
    chans = "<br>".join(
        f'<a href="{esc(c["link"])}" target="_blank">{esc(c["title"] or str(c["id"]))}</a>'
        + ("" if c["member"] is None else (" ✅" if c["member"] else " ➕"))
        for c in r["channels"])
    trs.append(f'<tr><td>{esc(r["prize"])}</td><td>{esc(r["until_date"][:16].replace("T", " "))} UTC</td>'
               f'<td>{chans}</td><td>{esc(r["status"] or "")}</td>'
               f'<td><a href="{esc(r["link"])}" target="_blank">открыть</a></td></tr>')
display(HTML(
    f"<p><b>Найдено активных розыгрышей: {len(rows)}</b> (✅ вы подписаны, ➕ нужно подписаться)</p>"
    "<table><tr><th>Приз</th><th>До</th><th>Каналы</th><th>Статус</th><th>Пост</th></tr>"
    + "".join(trs) + "</table>"))

try:
    from google.colab import files
    files.download("giveaways.csv")
except Exception:
    pass

## (Необязательно) Не вводить код при каждом запуске

Colab через какое-то время сбрасывает машину, и вход приходится повторять. Чтобы этого избежать, получите **строку сессии** и сохраните её в «Секреты» под именем `TG_STRING_SESSION`.

> ⚠️ Строка сессии даёт **полный доступ к аккаунту**. Сохраните её только в «Секреты», затем очистите вывод ячейки (⋮ → *Clear output*). Не публикуйте её и никому не передавайте.
> Если строка утекла, завершите этот сеанс в Telegram: *Настройки → Устройства*.

In [ ]:
# 5. (необязательно) показать строку сессии
from telethon.sessions import StringSession
import tg_giveaway_finder as tgf

client = tgf.make_client()
await client.start()
print(StringSession.save(client.session))
await client.disconnect()